## Configuration & File Paths

In [ ]:
import os
from pathlib import Path

# Get notebook directory and construct data path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
BASE_DATA_PATH = os.path.join(NOTEBOOK_DIR, "..", "DSA4264 Data")

# Input paths
COURSES_DIR = os.path.join(BASE_DATA_PATH, "NUS-SMU-SUTD Courses")
JOBS_PARQUET = os.path.join(BASE_DATA_PATH, "processed_jobs_dual_embeddings.parquet")

# Course CSV files
NUS_COURSES_CSV = os.path.join(COURSES_DIR, "nus_courses.csv")
SMU_COURSES_CSV = os.path.join(COURSES_DIR, "smu_courses.csv")
SUTD_COURSES_CSV = os.path.join(COURSES_DIR, "sutd_courses.csv")

# Output directories (relative to notebook directory)
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, "bertopic_visualizations_global")
UNIVERSITY_JOB_MATCHES_DIR = os.path.join(NOTEBOOK_DIR, "university_job_matches")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Data path: {BASE_DATA_PATH}")
print(f"Jobs parquet: {JOBS_PARQUET}")
print(f"Courses directory: {COURSES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Notebook directory: /Users/teresaliau/dsa4264/notebooks
Data path: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data
Jobs parquet: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/processed_jobs_dual_embeddings.parquet
Courses directory: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses
Output directory: /Users/teresaliau/dsa4264/notebooks/bertopic_visualizations_global


#BERTOPIC


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN


# The refined, lean fluff list
job_fluff = [
    "job", "description", "apply", "candidates", "interested", "hiring",
    "qualifications", "preferred", "required", "responsibilities",
    "equivalent", "company", "opportunities", "role", "title",
    "experience", "years", "year", "skills", "work", "working", "team",
    "teams", "knowledge", "ability", "strong", "using", "understanding",
    "including", "related", "key", "help", "plus"
]
all_stop_words = list(ENGLISH_STOP_WORDS) + job_fluff

df = pd.read_parquet(JOBS_PARQUET)
df = df.drop_duplicates(subset=['description']).copy()

# filter for full-time/permanent roles
allowed_types = {"Full Time", "Permanent", "Contract"}
def keeps_allowed_types(emp_types):
    try:
        return bool(set(emp_types) & allowed_types)
    except TypeError:
        return False
df = df[df['employmentTypes'].apply(keeps_allowed_types)].copy()

print(f"Total jobs to cluster: {len(df)}")

embedding_model = SentenceTransformer('all-mpnet-base-v2')
all_job_embeddings = np.stack(df['embedding_mpnet'].values)


def tune_bertopic_params(docs, embeddings, vectorizer_model, n_trials=10):
    # Uses Optuna to find the mathematically best UMAP and HDBSCAN parameters
    cleaned_docs = [str(doc).lower().split() for doc in docs]
    id2word = corpora.Dictionary(cleaned_docs)

    def objective(trial):
        n_neighbors = trial.suggest_int("n_neighbors", 15, 50)
        n_components = trial.suggest_int("n_components", 3, 10)
        min_cluster_size = trial.suggest_int("min_cluster_size", 10, 50)

        temp_umap = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=42)
        temp_hdbscan = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=5, metric='euclidean', cluster_selection_method='eom')

        temp_topic_model = BERTopic(
            umap_model=temp_umap,
            hdbscan_model=temp_hdbscan,
            vectorizer_model=vectorizer_model,
            language="english",
            calculate_probabilities=False # Kept False during tuning to save massive memory
        )

        try:
            topics, _ = temp_topic_model.fit_transform(docs, embeddings=embeddings)

            # Penalize if it completely fails to find clusters
            if len(temp_topic_model.get_topic_info()) < 3:
                return 0.0

            topics_words = [[word for word, _ in temp_topic_model.get_topic(topic_id)] for topic_id in temp_topic_model.get_topics() if topic_id != -1]
            if not topics_words:
                return 0.0

            # Calculate Coherence
            coherence_model = CoherenceModel(topics=topics_words, texts=cleaned_docs, dictionary=id2word, coherence='c_v')
            coherence_score = coherence_model.get_coherence()

            # Penalize high outlier ratios
            outlier_ratio = topics.count(-1) / len(topics)

            return coherence_score * (1.0 - outlier_ratio)

        except Exception as e:
            return 0.0

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"   -> Best params found: {study.best_params} (Score: {study.best_value:.4f})")
    return study.best_params

def process_entire_dataset():
    print(f"\n{'='*60}")
    print("Processing Entire Global Dataset")
    print(f"{'='*60}")

    docs = df['description'].tolist()
    selected_embeddings = all_job_embeddings

    # NLP Sub-models
    vectorizer_model = CountVectorizer(stop_words=all_stop_words, ngram_range=(1, 3))
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
    representation_model = [KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.8)]

    # Find best parameters
    best_params = tune_bertopic_params(docs, selected_embeddings, vectorizer_model, n_trials=10)

    # Build the optimized sub-models
    umap_model = UMAP(
        n_neighbors=best_params['n_neighbors'],
        n_components=best_params['n_components'],
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=5,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # Assemble final BERTopic model
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        calculate_probabilities=False, # Set to False for global run to prevent memory crash
        language="english"
    )

    # Fit the Model
    print("Fitting optimized global BERTopic model...")
    topics, _ = topic_model.fit_transform(docs, embeddings=selected_embeddings)

    topic_info = topic_model.get_topic_info()
    num_topics = len(topic_info) - 1 # Subtract 1 because Topic -1 is the outlier pile
    print(f"Discovered {num_topics} unique, optimized clusters across the entire market!\n")

    # Print a summary to Console
    for index, row in topic_info.iterrows():
        topic_id = row['Topic']
        count = row['Count']

        if topic_id == -1:
            print(f"  [Outliers] {count} jobs did not fit into a specific niche.")
            continue

        words = ", ".join(row['Representation'][:7])
        print(f"  Topic {topic_id:2d} ({count:4d} jobs) -> {words}")

    # Save Visualizations
    file_prefix = "global_market"

    if num_topics > 1:
        topic_model.visualize_topics().write_html(f"{OUTPUT_DIR}/{file_prefix}_distance_map.html")
        topic_model.visualize_hierarchy().write_html(f"{OUTPUT_DIR}/{file_prefix}_hierarchy.html")
        topic_model.visualize_barchart(top_n_topics=16).write_html(f"{OUTPUT_DIR}/{file_prefix}_barchart.html")
        topic_model.visualize_documents(docs, embeddings=selected_embeddings).write_html(f"{OUTPUT_DIR}/{file_prefix}_document_map.html")
        print(f"Saved global visualizations to /{OUTPUT_DIR}")
    else:
        print(f"Not enough topics found to generate visualizations.")

    # Save the fitted model
    topic_model.save("bertopic_global_model", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

    # Annotate the jobs dataframe with their cluster assignments and save
    df_annotated = df.copy()
    df_annotated["topic_id"] = topics

    # Attach human-readable topic labels
    topic_label_map = {}
    for _, row in topic_model.get_topic_info().iterrows():
        tid = row["Topic"]
        if tid == -1:
            topic_label_map[tid] = "Outlier"
        else:
            topic_label_map[tid] = ", ".join(row["Representation"][:3])
    df_annotated["topic_label"] = df_annotated["topic_id"].map(topic_label_map)

    df_annotated.to_parquet("jobs_with_topics.parquet", engine="pyarrow")
    print(f"Saved annotated jobs to 'jobs_with_topics.parquet'")
    print(f"Saved BERTopic model to 'bertopic_global_model/'")

    return topic_model

if __name__ == "__main__":
    topic_model = process_entire_dataset()
    print("Done, check your 'bertopic_visualizations_global' folder.")


Total jobs to cluster: 16686


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# The refined, lean fluff list
job_fluff = [
    "job", "description", "apply", "candidates", "interested", "hiring",
    "qualifications", "preferred", "required", "responsibilities",
    "equivalent", "company", "opportunities", "role", "title",
    "experience", "years", "year", "skills", "work", "working", "team",
    "teams", "knowledge", "ability", "strong", "using", "understanding",
    "including", "related", "key", "help", "plus"
]
all_stop_words = list(ENGLISH_STOP_WORDS) + job_fluff

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(JOBS_PARQUET)
df = df.drop_duplicates(subset=['description']).copy()

# filter for full-time/permanent roles
allowed_types = {"Full Time", "Permanent", "Contract"}
def keeps_allowed_types(emp_types):
    try:
        return bool(set(emp_types) & allowed_types)
    except TypeError:
        return False
df = df[df['employmentTypes'].apply(keeps_allowed_types)].copy()

print(f"Total jobs to cluster: {len(df)}")

embedding_model = SentenceTransformer('all-mpnet-base-v2')
all_job_embeddings = np.stack(df['embedding_mpnet'].values)


def tune_bertopic_params(docs, embeddings, vectorizer_model, n_trials=10):
    # Uses Optuna to find the mathematically best UMAP and HDBSCAN parameters
    cleaned_docs = [str(doc).lower().split() for doc in docs]
    id2word = corpora.Dictionary(cleaned_docs)

    def objective(trial):
        n_neighbors = trial.suggest_int("n_neighbors", 15, 50)
        n_components = trial.suggest_int("n_components", 3, 10)
        min_cluster_size = trial.suggest_int("min_cluster_size", 10, 50)

        temp_umap = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=42)
        temp_hdbscan = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=5, metric='euclidean', cluster_selection_method='eom')

        temp_topic_model = BERTopic(
            umap_model=temp_umap,
            hdbscan_model=temp_hdbscan,
            vectorizer_model=vectorizer_model,
            language="english",
            calculate_probabilities=False # Kept False during tuning to save massive memory
        )

        try:
            topics, _ = temp_topic_model.fit_transform(docs, embeddings=embeddings)

            # Penalize if it completely fails to find clusters
            if len(temp_topic_model.get_topic_info()) < 3:
                return 0.0

            topics_words = [[word for word, _ in temp_topic_model.get_topic(topic_id)] for topic_id in temp_topic_model.get_topics() if topic_id != -1]
            if not topics_words:
                return 0.0

            # Calculate Coherence
            coherence_model = CoherenceModel(topics=topics_words, texts=cleaned_docs, dictionary=id2word, coherence='c_v')
            coherence_score = coherence_model.get_coherence()

            # Penalize high outlier ratios
            outlier_ratio = topics.count(-1) / len(topics)

            return coherence_score * (1.0 - outlier_ratio)

        except Exception as e:
            return 0.0

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"   -> Best params found: {study.best_params} (Score: {study.best_value:.4f})")
    return study.best_params

def process_entire_dataset():
    print(f"\n{'='*60}")
    print("Processing Entire Global Dataset")
    print(f"{'='*60}")

    docs = df['description'].tolist()
    selected_embeddings = all_job_embeddings

    # NLP Sub-models
    vectorizer_model = CountVectorizer(stop_words=all_stop_words, ngram_range=(1, 3))
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
    representation_model = [KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.8)]

    # Find best parameters
    best_params = tune_bertopic_params(docs, selected_embeddings, vectorizer_model, n_trials=10)

    # Build the optimized sub-models
    umap_model = UMAP(
        n_neighbors=best_params['n_neighbors'],
        n_components=best_params['n_components'],
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=5,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # Assemble final BERTopic model
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        calculate_probabilities=False, # Set to False for global run to prevent memory crash
        language="english"
    )

    # Fit the Model
    print("Fitting optimized global BERTopic model...")
    topics, _ = topic_model.fit_transform(docs, embeddings=selected_embeddings)

    topic_info = topic_model.get_topic_info()
    num_topics = len(topic_info) - 1 # Subtract 1 because Topic -1 is the outlier pile
    print(f"Discovered {num_topics} unique, optimized clusters across the entire market!\n")

    # Print a summary to Console
    for index, row in topic_info.iterrows():
        topic_id = row['Topic']
        count = row['Count']

        if topic_id == -1:
            print(f"  [Outliers] {count} jobs did not fit into a specific niche.")
            continue

        words = ", ".join(row['Representation'][:7])
        print(f"  Topic {topic_id:2d} ({count:4d} jobs) -> {words}")

    # Save Visualizations
    file_prefix = "global_market"

    if num_topics > 1:
        topic_model.visualize_topics().write_html(f"{OUTPUT_DIR}/{file_prefix}_distance_map.html")
        topic_model.visualize_hierarchy().write_html(f"{OUTPUT_DIR}/{file_prefix}_hierarchy.html")
        topic_model.visualize_barchart(top_n_topics=16).write_html(f"{OUTPUT_DIR}/{file_prefix}_barchart.html")
        topic_model.visualize_documents(docs, embeddings=selected_embeddings).write_html(f"{OUTPUT_DIR}/{file_prefix}_document_map.html")
        print(f"Saved global visualizations to /{OUTPUT_DIR}")
    else:
        print(f"Not enough topics found to generate visualizations.")

    # Save the fitted model
    topic_model.save(os.path.join(NOTEBOOK_DIR, "bertopic_global_model"), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

    # Annotate the jobs dataframe with their cluster assignments and save
    df_annotated = df.copy()
    df_annotated["topic_id"] = topics

    # Attach human-readable topic labels
    topic_label_map = {}
    for _, row in topic_model.get_topic_info().iterrows():
        tid = row["Topic"]
        if tid == -1:
            topic_label_map[tid] = "Outlier"
        else:
            topic_label_map[tid] = ", ".join(row["Representation"][:3])
    df_annotated["topic_label"] = df_annotated["topic_id"].map(topic_label_map)

    df_annotated.to_parquet(os.path.join(NOTEBOOK_DIR, "jobs_with_topics.parquet"), engine="pyarrow")
    print(f"Saved annotated jobs to 'jobs_with_topics.parquet'")
    print(f"Saved BERTopic model to 'bertopic_global_model/'")

    return topic_model

if __name__ == "__main__":
    topic_model = process_entire_dataset()
    print("Done, check your 'bertopic_visualizations_global' folder.")

In [ ]:
embedding_model = SentenceTransformer('all-mpnet-base-v2')
topic_model = BERTopic.load("bertopic_global_model", embedding_model=embedding_model)

print("Extracting Topic Info...")
topic_info = topic_model.get_topic_info()

output_file = "global_topic_info.csv"
topic_info.to_csv(output_file, index=False)

print(f"Success! Saved all cluster definitions to '{output_file}'")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting Topic Info...
Success! Saved all cluster definitions to 'global_topic_info.csv'



# MODULE PIPELINE
Fetches full module descriptions from NUSMods API
and embeds them directly with MPNet.

# FLOW
- Load modules
- Load jobs annotated with BERTopic cluster IDs
- Push module embeddings through the BERTopic model → assigns each module to its nearest job cluster
- Count module votes per cluster → cluster relevance weight
- Keep only jobs in relevant clusters (BERTopic filter)
- Re-rank within shortlist using full cosine similarity so the best individual job wins
- Combined score = cluster weight + avg sim (similarity w all mods) + max sim (highest scoore w any module)


In [ ]:
OUTPUT_FILE = "all_nus_modules_embedded.parquet"
START_YEAR  = 2025

print("Loading MPNet embedding model...")
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# API fetcher
def fetch_module_data(mod_code, start_year):
    year = start_year
    while year >= 2015:
        url = f"https://api.nusmods.com/v2/{year}-{year+1}/modules/{mod_code.upper()}.json"
        r = requests.get(url)
        if r.status_code == 200:
            d = r.json()
            return {
                "code":          mod_code,
                "title":         d.get("title", ""),
                "description":   d.get("description", ""),
                "faculty":       d.get("faculty", ""),
                "department":    d.get("department", ""),
                "academic_year": f"{year}-{year+1}"
            }
        elif r.status_code == 404:
            year -= 1
        else:
            break
    return None


df = pd.read_csv(NUS_COURSES_CSV)
df.columns = df.columns.str.strip().str.lower()
df_unique = df.drop_duplicates(subset=["code"]).copy()

unique_modules = df_unique["code"].dropna().tolist()
print(f"Found {len(unique_modules)} unique modules across all majors.")

In [ ]:
OUTPUT_FILE = "all_nus_modules_embedded.parquet"
START_YEAR  = 2025

print("Loading MPNet embedding model...")
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# API fetcher
def fetch_module_data(mod_code, start_year):
    year = start_year
    while year >= 2015:
        url = f"https://api.nusmods.com/v2/{year}-{year+1}/modules/{mod_code.upper()}.json"
        r = requests.get(url)
        if r.status_code == 200:
            d = r.json()
            return {
                "code":          mod_code,
                "title":         d.get("title", ""),
                "description":   d.get("description", ""),
                "faculty":       d.get("faculty", ""),
                "department":    d.get("department", ""),
                "academic_year": f"{year}-{year+1}"
            }
        elif r.status_code == 404:
            year -= 1
        else:
            break
    return None


df = pd.read_csv(NUS_COURSES_CSV)
df.columns = df.columns.str.strip().str.lower()
df_unique = df.drop_duplicates(subset=["code"]).copy()

unique_modules = df_unique["code"].dropna().tolist()
print(f"Found {len(unique_modules)} unique modules across all majors.")

In [ ]:
api_results, missing = [], []
for i, code in enumerate(unique_modules):
    if i % 20 == 0 and i > 0:
        print(f"  {i}/{len(unique_modules)} done...")

    data = fetch_module_data(code, START_YEAR)

    if data:
        api_results.append(data)
    else:
        missing.append(code)

    time.sleep(0.1)

if missing:
    print(f"\n Could not find data for {len(missing)} modules. Examples: {missing[:5]}...")

if not api_results:
    raise RuntimeError("No module data fetched")


api_df = pd.DataFrame(api_results)

df_unique.columns = df_unique.columns.str.lower().str.strip()
api_df.columns = api_df.columns.str.lower().str.strip()

# merge the API descriptions back
final_df = pd.merge(df_unique, api_df, on="code", how="inner")

if "description" not in final_df.columns:
    if "description_y" in final_df.columns:
        final_df = final_df.rename(columns={"description_y": "description"})
    elif "description_x" in final_df.columns:
        final_df = final_df.rename(columns={"description_x": "description"})
    else:
        raise RuntimeError(f"No description column found. Columns: {final_df.columns.tolist()}")

# drop modules with no description
final_df["description"] = final_df["description"].fillna("").str.strip()
final_df = final_df[final_df["description"] != ""].reset_index(drop=True)

print(f"{len(final_df)} university-wide modules successfully prepped.")

# embed
embeddings = embedding_model.encode(
    final_df["description"].tolist(),
    show_progress_bar=True
)
final_df["skill_embedding"] = list(embeddings)

final_df.to_parquet(OUTPUT_FILE, engine="pyarrow")
print(f"\n Saved {len(final_df)} embedded modules to '{OUTPUT_FILE}'")
print("   Columns:", final_df.columns.tolist())

  20/381 done...
  40/381 done...
  60/381 done...
  80/381 done...
  100/381 done...
  120/381 done...
  140/381 done...
  160/381 done...
  180/381 done...
  200/381 done...
  220/381 done...
  240/381 done...
  260/381 done...
  280/381 done...
  300/381 done...
  320/381 done...
  340/381 done...
  360/381 done...
  380/381 done...

 Could not find data for 12 modules. Examples: ['BN2104', 'BN2404', 'BN3405', 'BN3406', 'CP2880']...
369 university-wide modules successfully prepped.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]


 Saved 369 embedded modules to 'all_nus_modules_embedded.parquet'
   Columns: ['course', 'type', 'code', 'description_x', 'weight', 'title', 'description', 'faculty', 'department', 'academic_year', 'skill_embedding']


SMU / SUTD

In [ ]:
smu_df = pd.read_csv(SMU_COURSES_CSV)
all_smu_modules_embedded = build_embedded_modules(
    df_unique=smu_df,
    output_file="all_smu_modules_embedded.parquet",
    embedding_model=embedding_model
)

In [ ]:
sutd_df = pd.read_csv(SUTD_COURSES_CSV)
all_sutd_modules_embedded = build_embedded_modules(
    df_unique=sutd_df,
    output_file="all_sutd_modules_embedded.parquet",
    embedding_model=embedding_model
)

In [ ]:
sutd_df = pd.read_csv(SUTD_COURSES_CSV)
all_sutd_modules_embedded = build_embedded_modules(
    df_unique=sutd_df,
    output_file="all_sutd_modules_embedded.parquet",
    embedding_model=embedding_model
)

# Matching to jobs

In [ ]:
os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

jobs_df = pd.read_parquet("jobs_with_topics.parquet")
jobs_clustered = jobs_df[jobs_df["topic_id"] != -1].copy().reset_index(drop=True)

try:
    topic_model
except NameError:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")
    topic_model = BERTopic.load("bertopic_global_model", embedding_model=embedding_model)


def run_university_matching(course_csv_path, modules_parquet_path, university_prefix):
    print(f"\n===== Processing {university_prefix.upper()} =====")

    csv_df = pd.read_csv(course_csv_path)
    csv_df.columns = csv_df.columns.str.strip().str.lower()

    all_courses = csv_df["course"].dropna().unique()
    all_modules_df = pd.read_parquet(modules_parquet_path).copy()

    # assign topic clusters to modules
    module_texts = all_modules_df["description"].fillna("").tolist()
    module_topics, _ = topic_model.transform(module_texts)
    all_modules_df["assigned_topic_id"] = module_topics

    topic_info = topic_model.get_topic_info()
    topic_label_map = {
        row["Topic"]: ", ".join(row["Representation"][:3])
        for _, row in topic_info.iterrows()
    }
    all_modules_df["assigned_topic_label"] = (
        all_modules_df["assigned_topic_id"]
        .map(topic_label_map)
        .fillna("Outlier")
    )

    for course_name in all_courses:
        required_codes = (
            csv_df[csv_df["course"] == course_name]["code"]
            .dropna()
            .astype(str)
            .unique()
        )

        degree_modules_df = all_modules_df[
            all_modules_df["code"].astype(str).isin(required_codes)
        ].copy()

        if len(degree_modules_df) == 0:
            print(f"Skipping '{course_name}' - No modules found in embedded dataset.")
            continue

        topic_module_counts = (
            degree_modules_df[degree_modules_df["assigned_topic_id"] != -1]
            .groupby(["assigned_topic_id", "assigned_topic_label"])
            .size()
            .reset_index(name="module_count")
            .sort_values("module_count", ascending=False)
        )

        if len(topic_module_counts) == 0:
            print(f"Skipping '{course_name}' - Modules only matched with Outliers.")
            continue

        relevant_topic_ids = topic_module_counts["assigned_topic_id"].tolist()
        topic_weight_map = dict(
            zip(topic_module_counts["assigned_topic_id"], topic_module_counts["module_count"])
        )

        matched_jobs = jobs_clustered[
            jobs_clustered["topic_id"].isin(relevant_topic_ids)
        ].copy()

        if len(matched_jobs) == 0:
            print(f"Skipping '{course_name}' - No matched jobs found.")
            continue

        matched_jobs["topic_module_count"] = (
            matched_jobs["topic_id"].map(topic_weight_map).fillna(0)
        )

        # cosine similarity
        course_embeddings = np.stack(degree_modules_df["skill_embedding"].values)
        job_desc_embeddings = np.stack(matched_jobs["embedding_mpnet"].values)

        sim_matrix = cosine_similarity(course_embeddings, job_desc_embeddings)

        matched_jobs["avg_similarity"] = sim_matrix.mean(axis=0)
        matched_jobs["max_similarity"] = sim_matrix.max(axis=0)

        top_module_idx = sim_matrix.argmax(axis=0)

        if "title" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} – {degree_modules_df.iloc[i]['title']}"
                for i in top_module_idx
            ]
        elif "type" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} ({degree_modules_df.iloc[i]['type']})"
                for i in top_module_idx
            ]
        else:
            matched_jobs["best_module"] = [
                str(degree_modules_df.iloc[i]["code"])
                for i in top_module_idx
            ]

        max_module_count = topic_module_counts["module_count"].max()
        matched_jobs["combined_score"] = (
            0.4 * (matched_jobs["topic_module_count"] / max_module_count) +
            0.4 * matched_jobs["avg_similarity"] +
            0.2 * matched_jobs["max_similarity"]
        )

        safe_course = str(course_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
        output_path = os.path.join(
            UNIVERSITY_JOB_MATCHES_DIR,
            f"{university_prefix}_{safe_course}_matches.csv"
        )

        output_cols = [
            "title",
            "companyName",
            "combined_score",
            "topic_label",
            "topic_module_count",
            "avg_similarity",
            "max_similarity",
            "best_module",
            "description"
        ]
        output_cols = [c for c in output_cols if c in matched_jobs.columns]

        top_matches = matched_jobs.sort_values("combined_score", ascending=False)
        top_matches[output_cols].to_csv(output_path, index=False)

        top_job_title = str(top_matches.iloc[0]["title"])[:35] if len(top_matches) > 0 else "N/A"
        print(f"{course_name[:30]:<30} | {len(degree_modules_df):>3} modules | Top Job: {top_job_title}")


# run for each university
run_university_matching(
    course_csv_path=NUS_COURSES_CSV,
    modules_parquet_path="all_nus_modules_embedded.parquet",
    university_prefix="nus"
)

run_university_matching(
    course_csv_path=SMU_COURSES_CSV,
    modules_parquet_path="all_smu_modules_embedded.parquet",
    university_prefix="smu"
)

run_university_matching(
    course_csv_path=SUTD_COURSES_CSV,
    modules_parquet_path="all_sutd_modules_embedded.parquet",
    university_prefix="sutd"
)

In [ ]:
os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

jobs_df = pd.read_parquet(os.path.join(NOTEBOOK_DIR, "jobs_with_topics.parquet"))
jobs_clustered = jobs_df[jobs_df["topic_id"] != -1].copy().reset_index(drop=True)

try:
    topic_model
except NameError:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")
    topic_model = BERTopic.load(os.path.join(NOTEBOOK_DIR, "bertopic_global_model"), embedding_model=embedding_model)


def run_university_matching(course_csv_path, modules_parquet_path, university_prefix):
    print(f"\n===== Processing {university_prefix.upper()} =====")

    csv_df = pd.read_csv(course_csv_path)
    csv_df.columns = csv_df.columns.str.strip().str.lower()

    all_courses = csv_df["course"].dropna().unique()
    all_modules_df = pd.read_parquet(modules_parquet_path).copy()

    # assign topic clusters to modules
    module_texts = all_modules_df["description"].fillna("").tolist()
    module_topics, _ = topic_model.transform(module_texts)
    all_modules_df["assigned_topic_id"] = module_topics

    topic_info = topic_model.get_topic_info()
    topic_label_map = {
        row["Topic"]: ", ".join(row["Representation"][:3])
        for _, row in topic_info.iterrows()
    }
    all_modules_df["assigned_topic_label"] = (
        all_modules_df["assigned_topic_id"]
        .map(topic_label_map)
        .fillna("Outlier")
    )

    for course_name in all_courses:
        required_codes = (
            csv_df[csv_df["course"] == course_name]["code"]
            .dropna()
            .astype(str)
            .unique()
        )

        degree_modules_df = all_modules_df[
            all_modules_df["code"].astype(str).isin(required_codes)
        ].copy()

        if len(degree_modules_df) == 0:
            print(f"Skipping '{course_name}' - No modules found in embedded dataset.")
            continue

        topic_module_counts = (
            degree_modules_df[degree_modules_df["assigned_topic_id"] != -1]
            .groupby(["assigned_topic_id", "assigned_topic_label"])
            .size()
            .reset_index(name="module_count")
            .sort_values("module_count", ascending=False)
        )

        if len(topic_module_counts) == 0:
            print(f"Skipping '{course_name}' - Modules only matched with Outliers.")
            continue

        relevant_topic_ids = topic_module_counts["assigned_topic_id"].tolist()
        topic_weight_map = dict(
            zip(topic_module_counts["assigned_topic_id"], topic_module_counts["module_count"])
        )

        matched_jobs = jobs_clustered[
            jobs_clustered["topic_id"].isin(relevant_topic_ids)
        ].copy()

        if len(matched_jobs) == 0:
            print(f"Skipping '{course_name}' - No matched jobs found.")
            continue

        matched_jobs["topic_module_count"] = (
            matched_jobs["topic_id"].map(topic_weight_map).fillna(0)
        )

        # cosine similarity
        course_embeddings = np.stack(degree_modules_df["skill_embedding"].values)
        job_desc_embeddings = np.stack(matched_jobs["embedding_mpnet"].values)

        sim_matrix = cosine_similarity(course_embeddings, job_desc_embeddings)

        matched_jobs["avg_similarity"] = sim_matrix.mean(axis=0)
        matched_jobs["max_similarity"] = sim_matrix.max(axis=0)

        top_module_idx = sim_matrix.argmax(axis=0)

        if "title" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} – {degree_modules_df.iloc[i]['title']}"
                for i in top_module_idx
            ]
        elif "type" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} ({degree_modules_df.iloc[i]['type']})"
                for i in top_module_idx
            ]
        else:
            matched_jobs["best_module"] = [
                str(degree_modules_df.iloc[i]["code"])
                for i in top_module_idx
            ]

        max_module_count = topic_module_counts["module_count"].max()
        matched_jobs["combined_score"] = (
            0.4 * (matched_jobs["topic_module_count"] / max_module_count) +
            0.4 * matched_jobs["avg_similarity"] +
            0.2 * matched_jobs["max_similarity"]
        )

        safe_course = str(course_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
        output_path = os.path.join(
            UNIVERSITY_JOB_MATCHES_DIR,
            f"{university_prefix}_{safe_course}_matches.csv"
        )

        output_cols = [
            "title",
            "companyName",
            "combined_score",
            "topic_label",
            "topic_module_count",
            "avg_similarity",
            "max_similarity",
            "best_module",
            "description"
        ]
        output_cols = [c for c in output_cols if c in matched_jobs.columns]

        top_matches = matched_jobs.sort_values("combined_score", ascending=False)
        top_matches[output_cols].to_csv(output_path, index=False)

        top_job_title = str(top_matches.iloc[0]["title"])[:35] if len(top_matches) > 0 else "N/A"
        print(f"{course_name[:30]:<30} | {len(degree_modules_df):>3} modules | Top Job: {top_job_title}")


# run for each university
run_university_matching(
    course_csv_path=NUS_COURSES_CSV,
    modules_parquet_path=os.path.join(NOTEBOOK_DIR, "all_nus_modules_embedded.parquet"),
    university_prefix="nus"
)

run_university_matching(
    course_csv_path=SMU_COURSES_CSV,
    modules_parquet_path=os.path.join(NOTEBOOK_DIR, "all_smu_modules_embedded.parquet"),
    university_prefix="smu"
)

run_university_matching(
    course_csv_path=SUTD_COURSES_CSV,
    modules_parquet_path=os.path.join(NOTEBOOK_DIR, "all_sutd_modules_embedded.parquet"),
    university_prefix="sutd"
)